In [3]:
import os
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from multiprocessing import Pool, cpu_count
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

# Configuration
THRESHOLDS = [0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10, 0]
PERCENTILES = [50, 80, 95]
MAX_CORES = 26
OUTPUT_DIR = "./5-similarity-on-scores"

# 明确指定要处理的文件路径
INPUT_FILES = [
    "./4-all-reactiondata-results/4-all-reactiondata_evaluated.csv",
    "./4-Exp-Reaction-data-results/Exp-Reaction-data_evaluated.csv"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 科研论文配色方案（更正式的颜色）
RESEARCH_COLORS = {
    'blue': '#1f77b4',      # Matplotlib蓝色
    'orange': '#ff7f0e',    # Matplotlib橙色
    'green': '#2ca02c',     # Matplotlib绿色
    'red': '#d62728',       # Matplotlib红色
    'purple': '#9467bd',    # Matplotlib紫色
    'brown': '#8c564b',     # Matplotlib棕色
    'pink': '#e377c2',      # Matplotlib粉色
    'gray': '#7f7f7f',      # 灰色
    'olive': '#bcbd22',     # 橄榄色
    'cyan': '#17becf'       # 青色
}

# 数据集名称映射
DATASET_NAME_MAP = {
    '4-all-reactiondata_evaluated': 'All data',
    'Exp-Reaction-data_evaluated': 'Experimental data'
}

# Function to convert SMILES to Morgan fingerprint
def smiles_to_fp(smiles, radius=2, nBits=2048):
    """Convert SMILES to Morgan fingerprint."""
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)

# Function to compute similarity between two fingerprints
def compute_similarity(fp1, fp2):
    """Compute Tanimoto similarity between two fingerprints."""
    if fp1 is None or fp2 is None:
        return 0.0
    return DataStructs.TanimotoSimilarity(fp1, fp2)

# Function to process a single row
def process_row(args):
    """Process a single row to compute similarities."""
    p_smiles, s1_smiles, s2_smiles, p_fp_dict, s1_fp_dict, s2_fp_dict = args
    
    # Get fingerprints from dictionaries
    p_fp = p_fp_dict.get(p_smiles)
    s1_fp = s1_fp_dict.get(s1_smiles)
    s2_fp = s2_fp_dict.get(s2_smiles)
    
    # Initialize similarity values
    sim_s1_p = 0.0
    sim_s2_p = 0.0
    max_sim = 0.0
    predicted_by = "None"
    
    # Compute S1-P similarity if both exist
    if p_fp is not None and s1_fp is not None:
        sim_s1_p = compute_similarity(p_fp, s1_fp)
    
    # Compute S2-P similarity if both exist
    if p_fp is not None and s2_fp is not None:
        sim_s2_p = compute_similarity(p_fp, s2_fp)
    
    # Determine maximum similarity and which reactant contributed
    if sim_s1_p > 0 or sim_s2_p > 0:
        if sim_s1_p >= sim_s2_p:
            max_sim = sim_s1_p
            predicted_by = "S1"
        else:
            max_sim = sim_s2_p
            predicted_by = "S2"
    
    return sim_s1_p, sim_s2_p, max_sim, predicted_by

def analyze_best_score_data(df, base_name):
    """专门分析pred_best_score=1的数据"""
    print("\n" + "="*50)
    print("ANALYZING PRED_BEST_SCORE = 1 DATA")
    print("="*50)
    
    # 筛选pred_best_score=1的数据
    best_score_1_df = df[df['pred_best_score'] == 1].copy()
    
    if len(best_score_1_df) == 0:
        print("No data with pred_best_score = 1 found.")
        return None
    
    total_best_score_1 = len(best_score_1_df)
    print(f"Total reactions with pred_best_score = 1: {total_best_score_1}")
    
    # 筛选出可预测的反应（相似度>0）
    predictable_best = best_score_1_df[best_score_1_df['max_similarity'] > 0].copy()
    predictable_count = len(predictable_best)
    
    print(f"\nPredictable reactions (similarity > 0): {predictable_count}")
    print(f"Predictable proportion: {predictable_count/total_best_score_1:.2%}")
    
    # 初始化统计字典
    best_score_stats = {
        'total_count': total_best_score_1,
        'predictable_count': predictable_count,
        'predictable_proportion': predictable_count/total_best_score_1 if total_best_score_1 > 0 else 0
    }
    
    if predictable_count > 0:
        # 1. 最高相似度区间分布（0.1间隔）
        similarity_bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        bin_labels = ['0-0.1', '0.1-0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5', 
                     '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0']
        
        # 使用cut函数进行分箱
        predictable_best['similarity_bin'] = pd.cut(
            predictable_best['max_similarity'], 
            bins=similarity_bins, 
            labels=bin_labels,
            include_lowest=True
        )
        
        bin_distribution = predictable_best['similarity_bin'].value_counts().sort_index()
        bin_percentages = (bin_distribution / predictable_count * 100).round(2)
        
        best_score_stats['similarity_distribution'] = {
            'bins': bin_labels,
            'counts': bin_distribution.to_dict(),
            'percentages': bin_percentages.to_dict()
        }
        
        print("\nSimilarity Distribution (0.1 intervals):")
        print("-" * 40)
        for bin_label in bin_labels:
            count = bin_distribution.get(bin_label, 0)
            percentage = bin_percentages.get(bin_label, 0)
            print(f"  {bin_label}: {count} reactions ({percentage:.1f}%)")
        
        # 2. 相似度平均值
        avg_similarity = predictable_best['max_similarity'].mean()
        median_similarity = predictable_best['max_similarity'].median()
        std_similarity = predictable_best['max_similarity'].std()
        
        best_score_stats['similarity_measures'] = {
            'mean': avg_similarity,
            'median': median_similarity,
            'std': std_similarity,
            'min': predictable_best['max_similarity'].min(),
            'max': predictable_best['max_similarity'].max(),
            'q1': predictable_best['max_similarity'].quantile(0.25),
            'q3': predictable_best['max_similarity'].quantile(0.75)
        }
        
        print(f"\nSimilarity Statistics:")
        print("-" * 40)
        print(f"  Mean similarity: {avg_similarity:.3f}")
        print(f"  Median similarity: {median_similarity:.3f}")
        print(f"  Standard deviation: {std_similarity:.3f}")
        
        # 3. 相似度阈值分布（>50%, >60%, >70%, >80%, >90%）
        threshold_stats = {}
        thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
        
        print(f"\nThreshold Analysis:")
        print("-" * 40)
        
        for threshold in thresholds:
            count_above = len(predictable_best[predictable_best['max_similarity'] >= threshold])
            proportion_total = count_above / total_best_score_1
            proportion_predictable = count_above / predictable_count if predictable_count > 0 else 0
            
            threshold_stats[threshold] = {
                'count': count_above,
                'proportion_of_total': proportion_total,
                'proportion_of_predictable': proportion_predictable
            }
            
            print(f"  ≥{threshold:.0%}:")
            print(f"    Count: {count_above} reactions")
            print(f"    Proportion of total (pred_best_score=1): {proportion_total:.2%}")
            print(f"    Proportion of predictable: {proportion_predictable:.2%}")
        
        best_score_stats['threshold_analysis'] = threshold_stats
        
        # 4. 按predicted_by分类的统计
        if 'predicted_by' in predictable_best.columns:
            predictor_stats = predictable_best.groupby('predicted_by').agg({
                'max_similarity': ['count', 'mean', 'median', 'std', 'min', 'max']
            }).round(3)
            
            predictor_stats.columns = ['_'.join(col).strip() for col in predictor_stats.columns.values]
            best_score_stats['by_predictor'] = predictor_stats.to_dict('index')
            
            print(f"\nStatistics by Predictor:")
            print("-" * 40)
            for predictor, stats_row in predictor_stats.iterrows():
                print(f"  {predictor}:")
                print(f"    Count: {int(stats_row['max_similarity_count'])}")
                print(f"    Mean: {stats_row['max_similarity_mean']:.3f}")
                print(f"    Median: {stats_row['max_similarity_median']:.3f}")
        
        # 5. 保存详细数据
        output_csv = os.path.join(OUTPUT_DIR, f"{base_name}_pred_best_score_1_analysis.csv")
        
        # 创建详细的分析数据框
        detailed_df = predictable_best.copy()
        # 选择要包含的列，确保列名正确
        available_columns = []
        for col in ['P', 'S1', 'S2', 'pred_best_score', 'sim_S1_P', 'sim_S2_P', 'max_similarity', 'predicted_by', 'similarity_bin']:
            if col in detailed_df.columns:
                available_columns.append(col)
        
        detailed_df = detailed_df[available_columns]
        
        # 添加阈值标记列
        for threshold in thresholds:
            detailed_df[f'similarity_geq_{int(threshold*100)}'] = (
                detailed_df['max_similarity'] >= threshold
            ).astype(int)
        
        detailed_df.to_csv(output_csv, index=False)
        print(f"\nSaved detailed pred_best_score=1 analysis to: {output_csv}")
    
    return best_score_stats

def create_combined_visualizations(all_best_score_data, output_name="combined_pred_best_score_1_analysis"):
    """为多个数据集的pred_best_score=1数据创建组合可视化图表 - 科研论文格式"""
    
    if len(all_best_score_data) == 0:
        print("No pred_best_score=1 data to visualize.")
        return
    
    # 设置科研论文的图表参数
    plt.rcParams.update({
        'font.size': 10,
        'font.family': 'Arial',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'legend.fontsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.1
    })
    
    # 创建两个子图，调整尺寸比例
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    
    # 获取数据集名称（使用映射后的名称）
    dataset_names = []
    original_names = list(all_best_score_data.keys())
    for name in original_names:
        # 应用名称映射
        display_name = DATASET_NAME_MAP.get(name, name)
        dataset_names.append(display_name)
    
    # 为每个数据集分配颜色
    dataset_colors = [RESEARCH_COLORS['blue'], RESEARCH_COLORS['orange'], 
                     RESEARCH_COLORS['green'], RESEARCH_COLORS['red'],
                     RESEARCH_COLORS['purple']]
    
    # 1. Similarity Range Distribution (条形图)
    ax1 = axes[0]
    
    # 准备数据
    similarity_bins = ['0-0.1', '0.1-0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5', 
                      '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0']
    
    x = np.arange(len(similarity_bins))
    width = 0.7 / len(dataset_names)  # 动态调整宽度
    
    all_percentages = []  # 收集所有百分比数据
    
    for i, (orig_name, data) in enumerate(all_best_score_data.items()):
        percentages = []
        for bin_label in similarity_bins:
            if 'similarity_distribution' in data and 'percentages' in data['similarity_distribution']:
                percentages.append(data['similarity_distribution']['percentages'].get(bin_label, 0))
            else:
                percentages.append(0)
        
        all_percentages.append(percentages)
        
        display_name = DATASET_NAME_MAP.get(orig_name, orig_name)
        offset = (i - (len(dataset_names) - 1) / 2) * width
        ax1.bar(x + offset, percentages, width, 
                label=display_name, color=dataset_colors[i % len(dataset_colors)], 
                alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax1.set_xlabel('Similarity Range', fontweight='bold')
    ax1.set_ylabel('Percentage (%)', fontweight='bold')
    ax1.set_title('(a) Similarity Range Distribution', fontweight='bold', fontsize=11)
    ax1.set_xticks(x)
    ax1.set_xticklabels(similarity_bins, rotation=45, ha='right')
    ax1.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    # 将图例放在右上角内部
    ax1.legend(title='Dataset', loc='upper right', frameon=True, 
               fancybox=True, framealpha=0.9, edgecolor='black')
    
    # 设置y轴范围
    max_percentage = max([max(p) for p in all_percentages]) if all_percentages else 100
    ax1.set_ylim(0, max_percentage * 1.15)
    
    # 2. Similarity by Predictor (箱线图)
    ax2 = axes[1]
    
    predictor_data_by_dataset = {}
    all_sim_data = []  # 收集所有相似度数据用于设置y轴范围
    positions = []
    labels = []
    box_colors = []
    box_data = []
    
    pos_counter = 0
    for i, (orig_name, data) in enumerate(all_best_score_data.items()):
        if 'by_predictor' in data:
            display_name = DATASET_NAME_MAP.get(orig_name, orig_name)
            color = dataset_colors[i % len(dataset_colors)]
            
            for predictor in ['S1', 'S2']:
                if predictor in data['by_predictor']:
                    key = f"{display_name}\n{predictor}"
                    # 这里我们无法直接获取原始数据，所以创建一个示例分布
                    # 在实际应用中，您可能需要存储原始相似度值
                    mean_sim = data['by_predictor'][predictor]['max_similarity_mean']
                    std_sim = data['by_predictor'][predictor]['max_similarity_std']
                    count = data['by_predictor'][predictor]['max_similarity_count']
                    
                    # 生成模拟数据用于箱线图
                    np.random.seed(42 + i)  # 使用不同的种子
                    simulated_data = np.random.normal(mean_sim, std_sim, min(count, 1000))
                    simulated_data = np.clip(simulated_data, 0, 1)
                    all_sim_data.extend(simulated_data)
                    
                    predictor_data_by_dataset[key] = simulated_data
                    box_data.append(simulated_data)
                    positions.append(pos_counter)
                    labels.append(key)
                    box_colors.append(color)
                    pos_counter += 1
    
    if box_data:
        # 绘制箱线图
        box_plot = ax2.boxplot(box_data, positions=positions, widths=0.5, 
                               patch_artist=True, showfliers=False,
                               medianprops={'color': 'black', 'linewidth': 1.5},
                               whiskerprops={'linewidth': 1},
                               capprops={'linewidth': 1})
        
        # 设置箱线图颜色
        for patch, color in zip(box_plot['boxes'], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
            patch.set_edgecolor('black')
            patch.set_linewidth(0.8)
        
        ax2.set_xlabel('Dataset and Predictor', fontweight='bold')
        ax2.set_ylabel('Maximum Similarity', fontweight='bold')
        ax2.set_title('(b) Similarity by Predictor', fontweight='bold', fontsize=11)
        ax2.grid(True, alpha=0.3, linestyle='--')
        
        # 设置x轴刻度和标签
        ax2.set_xticks(positions)
        ax2.set_xticklabels(labels, rotation=45, ha='right')
        
        # 设置y轴范围
        if all_sim_data:
            y_min = max(0, min(all_sim_data) - 0.05)
            y_max = min(1, max(all_sim_data) + 0.05)
            ax2.set_ylim(y_min, y_max)
            # 添加水平参考线
            ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
            ax2.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
            ax2.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    
    # 调整子图间距
    plt.tight_layout()
    
    # 保存图表
    plot_path = os.path.join(OUTPUT_DIR, f"{output_name}.png")
    plt.savefig(plot_path, dpi=600, bbox_inches='tight', pad_inches=0.05)
    
    # 同时保存PDF格式（科研论文常用）
    plot_pdf_path = os.path.join(OUTPUT_DIR, f"{output_name}.pdf")
    plt.savefig(plot_pdf_path, dpi=600, bbox_inches='tight', pad_inches=0.05)
    
    plt.close()
    print(f"Saved combined visualization to:")
    print(f"  PNG: {plot_path}")
    print(f"  PDF: {plot_pdf_path}")

# Main analysis function
def analyze_similarity(csv_file_path):
    """Perform similarity analysis on the provided CSV file."""
    
    # Load data
    df = pd.read_csv(csv_file_path)
    
    # 检查并标准化列名
    # 首先检查是否包含必需的列
    required_cols = ['P', 'S1', 'S2', 'pred_best_score']
    
    # 检查列名是否存在（包括可能的变体）
    available_cols = set(df.columns)
    
    # 如果缺少pred_best_score，检查是否有best_score
    if 'pred_best_score' not in df.columns and 'best_score' in df.columns:
        df = df.rename(columns={'best_score': 'pred_best_score'})
        print(f"Renamed 'best_score' column to 'pred_best_score'")
    
    # 再次检查是否所有必需的列都存在
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Available columns: {list(df.columns)}")
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Convert SMILES to fingerprints
    print("Converting SMILES to fingerprints...")
    
    # Create fingerprint dictionaries for unique SMILES
    unique_smiles = set()
    for col in ['P', 'S1', 'S2']:
        unique_smiles.update(df[col].dropna().unique())
    
    # Generate fingerprints in parallel
    with Pool(min(MAX_CORES, cpu_count())) as pool:
        fps = pool.map(smiles_to_fp, list(unique_smiles))
    
    # Create dictionaries for fast lookup
    fp_dict = dict(zip(unique_smiles, fps))
    
    # Create separate dictionaries for each column
    p_fp_dict = {smiles: fp_dict[smiles] for smiles in df['P'].dropna().unique() if smiles in fp_dict}
    s1_fp_dict = {smiles: fp_dict[smiles] for smiles in df['S1'].dropna().unique() if smiles in fp_dict}
    s2_fp_dict = {smiles: fp_dict[smiles] for smiles in df['S2'].dropna().unique() if smiles in fp_dict}
    
    # Prepare arguments for parallel processing
    print("Computing similarities...")
    args_list = []
    for idx, row in df.iterrows():
        args_list.append((
            row['P'], row['S1'], row['S2'],
            p_fp_dict, s1_fp_dict, s2_fp_dict
        ))
    
    # Compute similarities in parallel
    with Pool(min(MAX_CORES, cpu_count())) as pool:
        results = pool.map(process_row, args_list)
    
    # Add results to dataframe
    df['sim_S1_P'] = [r[0] for r in results]
    df['sim_S2_P'] = [r[1] for r in results]
    df['max_similarity'] = [r[2] for r in results]
    df['predicted_by'] = [r[3] for r in results]
    
    # Filter reactions that can be predicted (similarity > 0)
    predictable_df = df[df['max_similarity'] > 0].copy()
    
    # Perform statistical analysis on predictable reactions
    stats_results = {}
    
    if len(predictable_df) > 0:
        max_sim_values = predictable_df['max_similarity']
        
        # Basic statistics
        stats_results['basic_stats'] = {
            'count': len(predictable_df),
            'mean': np.mean(max_sim_values),
            'median': np.median(max_sim_values),
            'std': np.std(max_sim_values),
            'min': np.min(max_sim_values),
            'max': np.max(max_sim_values),
            'q1': np.percentile(max_sim_values, 25),
            'q3': np.percentile(max_sim_values, 75)
        }
        
        # Percentile statistics
        stats_results['percentiles'] = {}
        for p in PERCENTILES:
            stats_results['percentiles'][f'P{p}'] = np.percentile(max_sim_values, p)
        
        # Threshold analysis
        stats_results['threshold_counts'] = {}
        for thr in THRESHOLDS:
            count_above = len(predictable_df[predictable_df['max_similarity'] >= thr])
            stats_results['threshold_counts'][thr] = {
                'count': count_above,
                'proportion': count_above / len(predictable_df) if len(predictable_df) > 0 else 0
            }
        
        # Distribution by predicted_by
        stats_results['by_predictor'] = predictable_df.groupby('predicted_by')['max_similarity'].agg([
            'count', 'mean', 'median', 'std', 'min', 'max'
        ]).to_dict('index')
        
        # Binning analysis (0.1 intervals)
        bins = np.arange(0, 1.1, 0.1)
        digitized = np.digitize(max_sim_values, bins) - 1
        bin_counts = {}
        for i in range(len(bins) - 1):
            count = np.sum(digitized == i)
            if count > 0:
                bin_label = f"{bins[i]:.1f}-{bins[i+1]:.1f}"
                bin_counts[bin_label] = count
        stats_results['bin_distribution'] = bin_counts
        
        # Statistical tests (if enough data)
        if 'S1' in stats_results['by_predictor'] and 'S2' in stats_results['by_predictor']:
            s1_vals = predictable_df[predictable_df['predicted_by'] == 'S1']['max_similarity']
            s2_vals = predictable_df[predictable_df['predicted_by'] == 'S2']['max_similarity']
            
            if len(s1_vals) > 1 and len(s2_vals) > 1:
                t_stat, p_value = stats.ttest_ind(s1_vals, s2_vals, equal_var=False)
                stats_results['ttest'] = {
                    't_statistic': t_stat,
                    'p_value': p_value,
                    'significant': p_value < 0.05
                }
    
    # 分析pred_best_score=1的数据
    base_name = os.path.splitext(os.path.basename(csv_file_path))[0]
    best_score_stats = analyze_best_score_data(df, base_name)
    
    # 将pred_best_score=1的统计结果添加到总统计中
    if best_score_stats:
        stats_results['pred_best_score_1_analysis'] = best_score_stats
    
    # Save results
    # Save enriched dataframe
    output_csv = os.path.join(OUTPUT_DIR, f"{base_name}_with_similarity.csv")
    df.to_csv(output_csv, index=False)
    print(f"Saved enriched data to: {output_csv}")
    
    # Save predictable reactions separately
    if len(predictable_df) > 0:
        predictable_csv = os.path.join(OUTPUT_DIR, f"{base_name}_predictable_reactions.csv")
        predictable_df.to_csv(predictable_csv, index=False)
        print(f"Saved predictable reactions to: {predictable_csv}")
    
    # Save statistics
    stats_output = os.path.join(OUTPUT_DIR, f"{base_name}_similarity_statistics.xlsx")
    with pd.ExcelWriter(stats_output, engine='openpyxl') as writer:
        # Basic statistics
        basic_stats_df = pd.DataFrame([stats_results.get('basic_stats', {})])
        basic_stats_df.to_excel(writer, sheet_name='Basic_Statistics', index=False)
        
        # Percentiles
        percentiles_df = pd.DataFrame(list(stats_results.get('percentiles', {}).items()),
                                     columns=['Percentile', 'Value'])
        percentiles_df.to_excel(writer, sheet_name='Percentiles', index=False)
        
        # Threshold analysis
        threshold_data = []
        for thr, data in stats_results.get('threshold_counts', {}).items():
            threshold_data.append({
                'Threshold': thr,
                'Count': data['count'],
                'Proportion': data['proportion']
            })
        threshold_df = pd.DataFrame(threshold_data)
        threshold_df.to_excel(writer, sheet_name='Threshold_Analysis', index=False)
        
        # Predictor distribution
        if 'by_predictor' in stats_results:
            predictor_df = pd.DataFrame(stats_results['by_predictor']).T
            predictor_df.to_excel(writer, sheet_name='Predictor_Distribution')
        
        # Bin distribution
        if 'bin_distribution' in stats_results:
            bin_df = pd.DataFrame(list(stats_results['bin_distribution'].items()),
                                 columns=['Bin', 'Count'])
            bin_df['Proportion'] = bin_df['Count'] / bin_df['Count'].sum()
            bin_df.to_excel(writer, sheet_name='Bin_Distribution', index=False)
        
        # Statistical test results
        if 'ttest' in stats_results:
            ttest_df = pd.DataFrame([stats_results['ttest']])
            ttest_df.to_excel(writer, sheet_name='Statistical_Tests', index=False)
        
        # pred_best_score=1 analysis
        if 'pred_best_score_1_analysis' in stats_results:
            best_stats = stats_results['pred_best_score_1_analysis']
            
            # 总体统计
            overall_df = pd.DataFrame([{
                'Total_reactions_pred_best_score_1': best_stats.get('total_count', 0),
                'Predictable_reactions': best_stats.get('predictable_count', 0),
                'Predictable_proportion': best_stats.get('predictable_proportion', 0)
            }])
            overall_df.to_excel(writer, sheet_name='PredBestScore1_Overview', index=False)
            
            # 相似度统计
            if 'similarity_measures' in best_stats:
                sim_df = pd.DataFrame([best_stats['similarity_measures']])
                sim_df.to_excel(writer, sheet_name='PredBestScore1_SimilarityStats', index=False)
            
            # 相似度分布
            if 'similarity_distribution' in best_stats:
                dist_data = []
                for bin_label in best_stats['similarity_distribution']['bins']:
                    dist_data.append({
                        'Similarity_Range': bin_label,
                        'Count': best_stats['similarity_distribution']['counts'].get(bin_label, 0),
                        'Percentage': best_stats['similarity_distribution']['percentages'].get(bin_label, 0)
                    })
                dist_df = pd.DataFrame(dist_data)
                dist_df.to_excel(writer, sheet_name='PredBestScore1_Distribution', index=False)
            
            # 阈值分析
            if 'threshold_analysis' in best_stats:
                thresh_data = []
                for thr, data in best_stats['threshold_analysis'].items():
                    thresh_data.append({
                        'Threshold': f'≥{int(thr*100)}%',
                        'Count': data['count'],
                        'Proportion_of_Total': data['proportion_of_total'],
                        'Proportion_of_Predictable': data['proportion_of_predictable']
                    })
                thresh_df = pd.DataFrame(thresh_data)
                thresh_df.to_excel(writer, sheet_name='PredBestScore1_Thresholds', index=False)
            
            # 按预测者分类
            if 'by_predictor' in best_stats:
                pred_df = pd.DataFrame(best_stats['by_predictor']).T
                pred_df.to_excel(writer, sheet_name='PredBestScore1_ByPredictor')
    
    print(f"Saved statistics to: {stats_output}")
    
    return df, stats_results

# Main execution
if __name__ == "__main__":
    # 检查输入文件是否存在
    csv_files = []
    missing_files = []
    
    for file_path in INPUT_FILES:
        if os.path.exists(file_path):
            csv_files.append(file_path)
            file_name = os.path.basename(file_path)
            file_size = os.path.getsize(file_path) / 1024  # Size in KB
            print(f"✓ Found: {file_name} ({file_size:.1f} KB)")
        else:
            missing_files.append(file_path)
            print(f"✗ Missing: {file_path}")
    
    if missing_files:
        print(f"\nWarning: {len(missing_files)} file(s) are missing.")
    
    if not csv_files:
        print("No valid CSV files found. Exiting.")
        exit(1)
    
    print(f"\nAnalyzing {len(csv_files)} CSV file(s)...")
    
    all_best_score_stats = {}
    
    try:
        for csv_file_path in csv_files:
            file_name = os.path.basename(csv_file_path)
            base_name = os.path.splitext(file_name)[0]
            display_name = DATASET_NAME_MAP.get(base_name, base_name)
            
            print(f"\n{'='*60}")
            print(f"Starting similarity analysis for: {file_name} ({display_name})")
            print(f"{'='*60}")
            
            # 检查文件是否存在且可读
            if not os.path.exists(csv_file_path):
                print(f"Error: File does not exist: {csv_file_path}")
                continue
            
            try:
                # 读取前几行以检查列名
                df_preview = pd.read_csv(csv_file_path, nrows=5)
                print(f"File preview - Columns: {list(df_preview.columns)}")
                
                enriched_df, statistics = analyze_similarity(csv_file_path)
                
                # 保存pred_best_score=1的统计数据
                if 'pred_best_score_1_analysis' in statistics:
                    all_best_score_stats[base_name] = statistics['pred_best_score_1_analysis']
                
                # Print summary
                print(f"\n{'='*50}")
                print(f"SUMMARY FOR {display_name}")
                print(f"{'='*50}")
                
                total_reactions = len(enriched_df)
                predictable_reactions = len(enriched_df[enriched_df['max_similarity'] > 0])
                
                print(f"Total reactions: {total_reactions}")
                print(f"Predictable reactions (similarity > 0): {predictable_reactions}")
                print(f"Predictable proportion: {predictable_reactions/total_reactions:.2%}")
                
                if predictable_reactions > 0:
                    max_sim_vals = enriched_df[enriched_df['max_similarity'] > 0]['max_similarity']
                    print(f"\nMaximum similarity statistics:")
                    print(f"  Mean: {np.mean(max_sim_vals):.3f}")
                    print(f"  Median: {np.median(max_sim_vals):.3f}")
                    print(f"  Std: {np.std(max_sim_vals):.3f}")
                    
                    # Predictor breakdown
                    predictor_counts = enriched_df[enriched_df['max_similarity'] > 0]['predicted_by'].value_counts()
                    print(f"\nPredictor breakdown:")
                    for predictor, count in predictor_counts.items():
                        if predictor != "None":
                            print(f"  {predictor}: {count} reactions ({count/predictable_reactions:.2%})")
                    
                    # Threshold summary
                    print(f"\nThreshold analysis:")
                    if 'threshold_counts' in statistics:
                        for thr, data in statistics['threshold_counts'].items():
                            print(f"  ≥{thr}: {data['count']} reactions ({data['proportion']:.2%})")
                
                # 打印pred_best_score=1的统计摘要
                if 'pred_best_score_1_analysis' in statistics:
                    best_stats = statistics['pred_best_score_1_analysis']
                    print(f"\n{'='*50}")
                    print(f"PRED_BEST_SCORE = 1 SUMMARY FOR {display_name}")
                    print(f"{'='*50}")
                    print(f"Total reactions with pred_best_score=1: {best_stats.get('total_count', 0)}")
                    print(f"Predictable reactions: {best_stats.get('predictable_count', 0)}")
                    print(f"Predictable proportion: {best_stats.get('predictable_proportion', 0):.2%}")
                    
                    if 'similarity_measures' in best_stats:
                        print(f"\nSimilarity statistics for pred_best_score=1:")
                        print(f"  Mean: {best_stats['similarity_measures']['mean']:.3f}")
                        print(f"  Median: {best_stats['similarity_measures']['median']:.3f}")
                    
                    if 'threshold_analysis' in best_stats:
                        print(f"\nThreshold analysis for pred_best_score=1:")
                        for thr, data in best_stats['threshold_analysis'].items():
                            print(f"  ≥{int(thr*100)}%: {data['count']} reactions ({data['proportion_of_predictable']:.2%})")
                
                print(f"\nResults for {display_name} saved to directory: {OUTPUT_DIR}")
                
            except Exception as e:
                print(f"Error processing {file_name}: {str(e)}")
                import traceback
                traceback.print_exc()
                continue
        
        # 创建组合可视化图表
        if all_best_score_stats:
            print(f"\n{'='*60}")
            print("CREATING COMBINED VISUALIZATIONS")
            print(f"{'='*60}")
            create_combined_visualizations(all_best_score_stats)
        
        print(f"\n{'='*60}")
        print("ANALYSIS COMPLETE")
        print(f"{'='*60}")
        print(f"Analyzed {len(csv_files)} CSV file(s)")
        print(f"All results saved to directory: {OUTPUT_DIR}")
        
        # Show saved files
        print("\nGenerated files:")
        saved_files = os.listdir(OUTPUT_DIR)
        for file in sorted(saved_files):
            file_path = os.path.join(OUTPUT_DIR, file)
            file_size = os.path.getsize(file_path) / 1024  # Size in KB
            print(f"  • {file} ({file_size:.1f} KB)")
        
        print(f"\nTotal files generated: {len(saved_files)}")
        
    except Exception as e:
        print(f"Error during analysis: {str(e)}")
        import traceback
        traceback.print_exc()

✓ Found: 4-all-reactiondata_evaluated.csv (216.6 KB)
✓ Found: Exp-Reaction-data_evaluated.csv (26.8 KB)

Analyzing 2 CSV file(s)...

Starting similarity analysis for: 4-all-reactiondata_evaluated.csv (All data)
File preview - Columns: ['Reaction_type', 'P', 'S1', 'S2', 'pred_best_methods', 'pred_best_methods_str', 'pred_best_score', 'error', 'pred_AM-I_P', 'pred_AM-I_S1', 'pred_AM-I_S2', 'pred_score_AM-I', 'pred_score_AM-II', 'pred_score_AM-III', 'pred_score_AM-IV', 'pred_score_AM-V', 'pred_score_AM-VI', 'pred_AM-VI_P', 'pred_AM-VI_S1', 'pred_AM-VI_S2', 'pred_AM-III_P', 'pred_AM-III_S1', 'pred_AM-III_S2', 'pred_AM-V_P', 'pred_AM-V_S1', 'pred_AM-V_S2', 'pred_AM-IV_P', 'pred_AM-IV_S1', 'pred_AM-IV_S2', 'pred_AM-II_P', 'pred_AM-II_S1', 'pred_AM-II_S2']
Converting SMILES to fingerprints...


[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerator
[20:05:56] DEPRECATION WARNING: please use MorganGenerat

Computing similarities...

ANALYZING PRED_BEST_SCORE = 1 DATA
Total reactions with pred_best_score = 1: 535

Predictable reactions (similarity > 0): 535
Predictable proportion: 100.00%

Similarity Distribution (0.1 intervals):
----------------------------------------
  0-0.1: 0 reactions (0.0%)
  0.1-0.2: 1 reactions (0.2%)
  0.2-0.3: 4 reactions (0.8%)
  0.3-0.4: 139 reactions (26.0%)
  0.4-0.5: 283 reactions (52.9%)
  0.5-0.6: 93 reactions (17.4%)
  0.6-0.7: 15 reactions (2.8%)
  0.7-0.8: 0 reactions (0.0%)
  0.8-0.9: 0 reactions (0.0%)
  0.9-1.0: 0 reactions (0.0%)

Similarity Statistics:
----------------------------------------
  Mean similarity: 0.450
  Median similarity: 0.442
  Standard deviation: 0.071

Threshold Analysis:
----------------------------------------
  ≥50%:
    Count: 135 reactions
    Proportion of total (pred_best_score=1): 25.23%
    Proportion of predictable: 25.23%
  ≥60%:
    Count: 17 reactions
    Proportion of total (pred_best_score=1): 3.18%
    Proporti

[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerator
[20:05:58] DEPRECATION WARNING: please use MorganGenerat

Computing similarities...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.



ANALYZING PRED_BEST_SCORE = 1 DATA
Total reactions with pred_best_score = 1: 48

Predictable reactions (similarity > 0): 48
Predictable proportion: 100.00%

Similarity Distribution (0.1 intervals):
----------------------------------------
  0-0.1: 0 reactions (0.0%)
  0.1-0.2: 0 reactions (0.0%)
  0.2-0.3: 2 reactions (4.2%)
  0.3-0.4: 18 reactions (37.5%)
  0.4-0.5: 21 reactions (43.8%)
  0.5-0.6: 5 reactions (10.4%)
  0.6-0.7: 2 reactions (4.2%)
  0.7-0.8: 0 reactions (0.0%)
  0.8-0.9: 0 reactions (0.0%)
  0.9-1.0: 0 reactions (0.0%)

Similarity Statistics:
----------------------------------------
  Mean similarity: 0.427
  Median similarity: 0.414
  Standard deviation: 0.077

Threshold Analysis:
----------------------------------------
  ≥50%:
    Count: 8 reactions
    Proportion of total (pred_best_score=1): 16.67%
    Proportion of predictable: 16.67%
  ≥60%:
    Count: 2 reactions
    Proportion of total (pred_best_score=1): 4.17%
    Proportion of predictable: 4.17%
  ≥70%:
  

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Saved combined visualization to:
  PNG: ./5-similarity-on-scores/combined_pred_best_score_1_analysis.png
  PDF: ./5-similarity-on-scores/combined_pred_best_score_1_analysis.pdf

ANALYSIS COMPLETE
Analyzed 2 CSV file(s)
All results saved to directory: ./5-similarity-on-scores

Generated files:
  • 4-all-reactiondata_evaluated_pred_best_score_1_analysis.csv (81.2 KB)
  • 4-all-reactiondata_evaluated_predictable_reactions.csv (255.8 KB)
  • 4-all-reactiondata_evaluated_similarity_statistics.xlsx (11.3 KB)
  • 4-all-reactiondata_evaluated_with_similarity.csv (255.8 KB)
  • Exp-Reaction-data_evaluated_pred_best_score_1_analysis.csv (7.5 KB)
  • Exp-Reaction-data_evaluated_predictable_reactions.csv (30.9 KB)
  • Exp-Reaction-data_evaluated_similarity_statistics.xlsx (11.2 KB)
  • Exp-Reaction-data_evaluated_with_similarity.csv (30.9 KB)
  • combined_pred_best_score_1_analysis.pdf (23.8 KB)
  • combined_pred_best_score_1_analysis.png (450.0 KB)

Total files generated: 10
